# Capstone 1 — Twitter Customer Support Routing

*Module 1 capstone — ML & NLP by Data Trainers LLC.*

Welcome to the Module 1 capstone. You have spent three notebooks building the classical NLP toolkit: topic modelling and NER, text classification, and logistic regression / boosting. Now you will put it all together on a **real, messy, noisy** dataset from a production company.

## The Situation

You are a data scientist at a large Twitter customer support team. Every day, thousands of customers tweet at companies — **@AppleSupport my phone won't turn on**, **@AmazonHelp my package didn't arrive**, **@SpotifyCares premium keeps disconnecting**, **@Uber_Support driver cancelled last minute**. Right now, each tweet is manually routed to the correct support queue, which costs time and money.

Management hands you 2.8M historical tweets from the **Twitter Customer Support (TWCS)** dataset and asks: *can you build an automated router that reads the tweet text and predicts which company it should go to?*

## The Task

Build a 4-class classifier (Apple / Amazon / Spotify / Uber) that predicts the target company from the tweet text alone, and compare two approaches head-to-head:

1. **Classical baseline** — TF-IDF + Logistic Regression (this is Notebook 3 territory).
2. **Neural model** — your first PyTorch classifier: `nn.Embedding` → masked mean-pool → `nn.Linear` → ReLU → `nn.Linear(4)`.

## The Twist — Labels Are Hidden

The dataset doesn't come with a `company` column. You will have to **extract the label from the reply metadata**: for each inbound customer tweet, look up its first reply, and use the reply's `author_id` as the label. This is *real* data engineering — production labels are almost never gift-wrapped.

## Learning Objectives

By the end of this capstone you can:

1. Frame a messy real-world problem as supervised classification.
2. Extract labels from implicit signals (reply metadata → inbound tweet).
3. Clean noisy social-media text (handles, URLs, hashtags).
4. Build a TF-IDF + Logistic Regression baseline with class weighting.
5. Write your first PyTorch classifier: `Embedding` → masked mean-pool → MLP.
6. Write a PyTorch training loop from scratch (no Keras `.fit()`).
7. Compare classical vs. neural on the same task with the same metrics.
8. Diagnose class imbalance and mitigate it with `class_weight='balanced'` / weighted `CrossEntropyLoss`.

**Runtime**: ~45 minutes end-to-end on a Colab GPU. CPU works too, just slower on the neural part.

**Prerequisites**: Notebooks 1, 2, 3.

## Section 0: Environment Setup

This notebook uses **PyTorch** (not Keras/TF). Run the next cell once per Colab session.

In [ ]:
# Install required packages (skip if you already have them locally)
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn textblob

In [ ]:
# Core imports
import os, re, random, math, time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")
print("Environment ready.")

## Section 1: Load the TWCS Dataset

The **Twitter Customer Support (TWCS)** corpus is 2.8M tweets between customers and company support accounts. We will use a 100K-row subsample for speed — the methodology scales up unchanged.

Each row has:

| Column | Meaning |
|--------|---------|
| `tweet_id` | Unique id for the tweet |
| `author_id` | Either a username (company, e.g. `AppleSupport`) or a numeric string (customer) |
| `inbound` | `True` if from customer, `False` if from company |
| `created_at` | Timestamp |
| `text` | The tweet text |
| `response_tweet_id` | Comma-separated list of reply `tweet_id`s, or empty |
| `in_response_to_tweet_id` | If this tweet is a reply, the `tweet_id` it's replying to |

In [ ]:
# Download the dataset (~170MB). Skip if you already have twcs.csv locally.
%%writefile get_capstone_data.sh
if [ ! -f twcs.csv ]; then
  wget -O twcs.csv "https://www.dropbox.com/scl/fi/6w2dchvsthwdol934nprt/twcs.csv?rlkey=u5205iehhrog88qt8iwm4udxs&dl=1"
fi

In [ ]:
!bash get_capstone_data.sh

In [ ]:
# Load the full CSV but cap to 100K rows for speed. The methodology is identical on full data.
SUBSET_SIZE = 100_000
twcs = pd.read_csv('./twcs.csv', nrows=SUBSET_SIZE)
print(f"Loaded {len(twcs):,} tweets")
print(f"Columns: {list(twcs.columns)}")
twcs.head()

In [ ]:
# Quick sanity check: inbound vs outbound split
print("inbound value counts:")
print(twcs['inbound'].value_counts())
print("\nTop 10 author_ids (companies dominate the outbound side):")
print(twcs['author_id'].value_counts().head(10))

## Section 2: Extract Labels — The Key Trick

**This is the most important cell in the notebook.** Re-read it twice.

The dataset has no `company` column. But it has something equivalent: the *reply* to each customer tweet was authored by the company that handled it. So:

```
Customer tweet (inbound=True)   ←—— we want to classify THIS
        ↑
        │ in_response_to_tweet_id
        │
Company reply (inbound=False)   ←—— its author_id IS the label
```

**Plan**:

1. Split the dataframe into `inbound` (customer) tweets and `outbound` (company) replies.
2. Build a map `{inbound_tweet_id: reply_author_id}` by iterating replies and looking at their `in_response_to_tweet_id`.
3. Attach that author_id as the label of each inbound tweet.
4. Keep only inbound tweets whose label is in our **top-4 companies**:

    `{AppleSupport, AmazonHelp, SpotifyCares, Uber_Support}`

Expect the result to be class-imbalanced — AppleSupport dominates. We will handle that with class weighting.

In [ ]:
# The four companies we want to route to.
TOP_AUTHORS = ['AppleSupport', 'AmazonHelp', 'SpotifyCares', 'Uber_Support']
print(f"Target classes: {TOP_AUTHORS}")

In [ ]:
# Step 1: separate inbound (customer) from outbound (company replies)
inbound = twcs[twcs['inbound'] == True].copy()
outbound = twcs[twcs['inbound'] == False].copy()
print(f"inbound (customer tweets):  {len(inbound):,}")
print(f"outbound (company replies): {len(outbound):,}")

In [ ]:
# Step 2: build {in_response_to_tweet_id -> author_id} map from outbound replies.
# We only care about replies whose author_id is one of our 4 companies.
outbound_targeted = outbound[outbound['author_id'].isin(TOP_AUTHORS)]

# in_response_to_tweet_id is a float in some rows (NaN); drop those.
outbound_targeted = outbound_targeted.dropna(subset=['in_response_to_tweet_id'])
outbound_targeted['in_response_to_tweet_id'] = outbound_targeted['in_response_to_tweet_id'].astype('int64')

reply_map = dict(zip(outbound_targeted['in_response_to_tweet_id'], outbound_targeted['author_id']))
print(f"Built reply_map with {len(reply_map):,} entries.")
print("Sample:", dict(list(reply_map.items())[:3]))

In [ ]:
# Step 3: attach the label to each inbound tweet, keep only labeled ones.
inbound['label'] = inbound['tweet_id'].map(reply_map)
labeled = inbound.dropna(subset=['label']).copy()
labeled = labeled[['tweet_id', 'text', 'label']].reset_index(drop=True)
print(f"Labeled tweets: {len(labeled):,}")
labeled.head(10)

In [ ]:
# Step 4: class distribution. Plot to visualize imbalance.
class_counts = labeled['label'].value_counts()
print(class_counts)

plt.figure(figsize=(7, 3.5))
class_counts.plot(kind='bar', color=['#5B8DEF', '#FF9F40', '#4ECDC4', '#FFD166'])
plt.title('Tweets per target company (before balancing)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Lab 1: Cap per-class size

The imbalance above will bias our classifiers. A simple mitigation — alongside model-level `class_weight` — is to **cap each class at 25K examples**. This still gives us up to 100K labeled tweets but keeps the dominant class from drowning everyone else.

**Your task**: sample at most `PER_CLASS_CAP` rows per class, then concatenate and shuffle.

Hints:

- `labeled.groupby('label').apply(lambda g: g.sample(...))` is the pandas idiom.
- Use `min(len(g), PER_CLASS_CAP)` so small classes aren't broken by `sample(n=...)` asking for too many.
- Shuffle with `.sample(frac=1, random_state=SEED).reset_index(drop=True)`.

In [ ]:
PER_CLASS_CAP = 25_000

# 1. For each class group in `labeled`, sample up to PER_CLASS_CAP rows.
balanced = None  # YOUR CODE

# 2. Shuffle the combined dataframe.
balanced = None  # YOUR CODE

# Verification
if balanced is not None:
    print(f"Balanced dataset size: {len(balanced):,}")
    print(balanced['label'].value_counts())

## Section 3: Clean the Tweets

Tweets are **extremely noisy**: `@handles`, `http://links`, hashtags, emojis, weird punctuation, ALL CAPS shouts. We need to clean them before tokenizing.

**Cleaning policy** (keep it simple):

1. Lowercase.
2. Remove URLs (`http\S+` and `t.co/...`).
3. Remove `@handles` (they are a dead giveaway for the label — leaking the answer!).
4. Keep the `#` symbol but drop the rest of non-alphanumerics.
5. Collapse whitespace.

> ⚠️ **Leakage warning**: If we keep `@AppleSupport` in the tweet text, the classifier will learn *just* to look at the handle and ignore the real content. Removing handles forces it to learn from the actual words — which is what we want in production too (the routing model should work even if the customer tweets plain English without tagging anyone).

### Demo: the `clean_tweet` function

In [ ]:
URL_RE = re.compile(r'http\S+|www\.\S+|t\.co/\S+')
HANDLE_RE = re.compile(r'@\w+')
NON_ALPHA_RE = re.compile(r"[^a-z0-9#\s]")
WS_RE = re.compile(r'\s+')

def clean_tweet(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = URL_RE.sub(' ', text)
    text = HANDLE_RE.sub(' ', text)
    text = NON_ALPHA_RE.sub(' ', text)
    text = WS_RE.sub(' ', text).strip()
    return text

# Quick sanity check on 3 real tweets
for ex in balanced['text'].iloc[:3] if balanced is not None else []:
    print('RAW :', ex)
    print('CLEAN:', clean_tweet(ex))
    print()

### Lab 2: Apply cleaning and split train/val/test

1. Apply `clean_tweet` to every row of `balanced['text']` and store in `balanced['clean']`.
2. Drop rows where the cleaned text is empty (some tweets were nothing but URLs/handles).
3. Do a **stratified** train/val/test split: 80% train, 10% val, 10% test. Stratify on `label`.

In [ ]:
# 1. Clean all tweets
balanced['clean'] = None  # YOUR CODE

# 2. Drop rows whose clean text is an empty string
balanced = None  # YOUR CODE (hint: boolean mask, then .reset_index(drop=True))

# 3. Stratified 80/10/10 split. First split off test, then split remainder into train/val.
X_all, y_all = balanced['clean'].values, balanced['label'].values
X_trainval, X_test, y_trainval, y_test = None, None, None, None  # YOUR CODE
X_train, X_val, y_train, y_val = None, None, None, None            # YOUR CODE

# Verification
if X_train is not None:
    print(f"Train: {len(X_train):,}   Val: {len(X_val):,}   Test: {len(X_test):,}")
    print("Train class distribution:", pd.Series(y_train).value_counts().to_dict())

## Section 4: Classical Baseline — TF-IDF + Logistic Regression

Before pulling out the neural hammer, always establish a baseline. A well-tuned **TF-IDF + Logistic Regression** often hits 70-80% macro-F1 on short-text classification and is the honest yardstick a neural model must beat.

Recipe:

1. `TfidfVectorizer(min_df=3, ngram_range=(1,2))` — unigrams + bigrams, drop hapaxes.
2. Fit on **training data only** (never on test!).
3. `LogisticRegression(class_weight='balanced', max_iter=1000)` — the `balanced` flag auto-weights inverse to class frequency, handling the imbalance from inside the loss.
4. Report `classification_report` with `digits=3` so you can see the macro-F1 clearly.

### Lab 3: Fit and evaluate the TF-IDF + LogReg baseline

In [ ]:
# 1. Build the vectorizer and fit on training text
tfidf = None  # YOUR CODE (TfidfVectorizer with min_df=3 and ngram_range=(1,2))
Xtr_tfidf = None  # YOUR CODE (fit_transform on X_train)
Xval_tfidf = None # YOUR CODE (transform on X_val)
Xte_tfidf = None  # YOUR CODE (transform on X_test)

# 2. Train LogReg with balanced class weights
logreg = None  # YOUR CODE
# logreg.fit(...)

# 3. Predict and evaluate on the test set
y_pred_lr = None  # YOUR CODE

if y_pred_lr is not None:
    print(classification_report(y_test, y_pred_lr, digits=3))

### Lab 4: Confusion matrix

The classification report tells us *macro* performance, but we also want to know: *where* does the model get confused? Are customers of Spotify ever mistakenly routed to Apple? Etc.

Compute the confusion matrix on the test set and plot it as a heatmap. Use `TOP_AUTHORS` (or the sorted unique labels) as axis labels.

In [ ]:
# 1. Compute the confusion matrix with the class order fixed to TOP_AUTHORS
cm = None  # YOUR CODE (confusion_matrix with labels=TOP_AUTHORS)

# 2. Plot it as a seaborn heatmap
if cm is not None:
    plt.figure(figsize=(5.5, 4.5))
    # YOUR CODE: sns.heatmap(cm, annot=True, fmt='d', ...)
    plt.title('LogReg baseline — confusion matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

## Section 5: Neural Classifier in PyTorch

Now the fun part: your first PyTorch classifier. Architecture:

```
token ids (B, T)  ──▶ Embedding(V, 64)  ──▶ (B, T, 64)
                                               │
                          masked mean over T  ▼
                                              (B, 64)
                                               │
                              Linear(64, 128) ▼  ──▶ ReLU ──▶ Dropout(0.3)
                                               │
                              Linear(128, 4)  ▼
                                              (B, 4)  ← logits, feed to CrossEntropyLoss
```

Key ingredients we'll need:

- **Vocabulary** mapping token → id (min_freq = 3).
- **Tokenizer** (simple whitespace split is fine after our cleaning).
- **Dataset + DataLoader** with a `collate_fn` that pads within each batch.
- **Masked mean pooling** — we must NOT average over the padding tokens!
- **Training loop** that we write ourselves (no `.fit()`).

### Demo: tokenizer, vocabulary, and label encoding

In [ ]:
# Hyperparameters for the neural model
VOCAB_MIN_FREQ = 3
MAX_LEN        = 40      # tweets are short; 40 tokens covers the vast majority
EMBEDDING_DIM  = 64
HIDDEN_DIM     = 128
BATCH_SIZE     = 64
EPOCHS         = 10
LR             = 1e-3
DROPOUT        = 0.3
PATIENCE       = 2       # early stopping on val F1

PAD_IDX, UNK_IDX = 0, 1

def tokenize(text):
    """Simple whitespace tokenizer — our cleaning already did the heavy lifting."""
    return text.split()

def build_vocab(texts, min_freq=VOCAB_MIN_FREQ):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    # ids 0 and 1 reserved for <pad> and <unk>
    vocab = {'<pad>': PAD_IDX, '<unk>': UNK_IDX}
    for token, freq in counter.most_common():
        if freq >= min_freq:
            vocab[token] = len(vocab)
    return vocab

vocab = build_vocab(X_train)
vocab_size = len(vocab)
print(f"Vocabulary size (min_freq={VOCAB_MIN_FREQ}): {vocab_size:,}")
print("Sample tokens:", list(vocab.items())[:10])

In [ ]:
# Label encoding: string label -> int index, in a FIXED order so nothing drifts.
LABEL2ID = {name: i for i, name in enumerate(TOP_AUTHORS)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}
print("LABEL2ID:", LABEL2ID)

def encode_text(text, max_len=MAX_LEN):
    ids = [vocab.get(tok, UNK_IDX) for tok in tokenize(text)[:max_len]]
    return ids  # variable length; we pad inside collate_fn, not here.

y_train_ids = np.array([LABEL2ID[l] for l in y_train])
y_val_ids   = np.array([LABEL2ID[l] for l in y_val])
y_test_ids  = np.array([LABEL2ID[l] for l in y_test])
print("Label id distribution (train):", np.bincount(y_train_ids))

### Demo: PyTorch Dataset + DataLoader with padding collate

In PyTorch, **you** write the Dataset. The Dataset yields individual examples; a `collate_fn` fuses a list of examples into a batch. Because sequences have different lengths, the collate_fn is also where we pad to the longest in the batch and build the attention mask.

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [encode_text(t) for t in texts]
        self.labels = labels.astype(np.int64)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])

def pad_collate(batch):
    """batch = [(ids_list, label_int), ...]
       Returns (ids_tensor (B, T), mask_tensor (B, T), y_tensor (B,))"""
    seqs, ys = zip(*batch)
    maxlen = max(1, max(len(s) for s in seqs))
    ids = torch.full((len(seqs), maxlen), PAD_IDX, dtype=torch.long)
    mask = torch.zeros((len(seqs), maxlen), dtype=torch.long)
    for i, s in enumerate(seqs):
        if len(s) == 0:  # safety net: encode an empty string -> one <unk> so model still sees something
            ids[i, 0] = UNK_IDX
            mask[i, 0] = 1
        else:
            ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)
            mask[i, :len(s)] = 1
    y = torch.tensor(ys, dtype=torch.long)
    return ids, mask, y

train_ds = TweetDataset(X_train, y_train_ids)
val_ds   = TweetDataset(X_val,   y_val_ids)
test_ds  = TweetDataset(X_test,  y_test_ids)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=pad_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate)

# Peek at one batch
ids_b, mask_b, y_b = next(iter(train_loader))
print("ids :", ids_b.shape, "mask:", mask_b.shape, "y:", y_b.shape)

### Lab 5: Build the `TweetClassifier` model

Implement the `__init__` and `forward` methods. Architecture (see the diagram above):

- `self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)`
- `self.fc1 = nn.Linear(emb_dim, hidden_dim)`
- `self.drop = nn.Dropout(dropout)`
- `self.fc2 = nn.Linear(hidden_dim, n_classes)`

In `forward(x, mask)`:

1. Embed `x` → shape `(B, T, E)`.
2. Apply the mask so padding tokens don't contribute to the mean.
    - `mask_f = mask.unsqueeze(-1).float()` — shape `(B, T, 1)`.
    - `e_mean = (e * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)`
3. FC(128) → ReLU → Dropout → FC(n_classes). Return logits (do NOT apply softmax — `CrossEntropyLoss` applies log-softmax internally).

> ⚠️ **Why masked mean?** If we did `e.mean(dim=1)` we'd divide by `T` including padding positions, diluting real tokens by however much padding was glued on. Masked mean divides only by the real token count.

In [ ]:
class TweetClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, n_classes, pad_idx=PAD_IDX, dropout=DROPOUT):
        super().__init__()
        # YOUR CODE: define self.emb, self.fc1, self.drop, self.fc2
        self.emb = None   # YOUR CODE
        self.fc1 = None   # YOUR CODE
        self.drop = None  # YOUR CODE
        self.fc2 = None   # YOUR CODE

    def forward(self, x, mask):
        # 1. Embed
        e = None          # YOUR CODE
        # 2. Masked mean pool
        e_mean = None     # YOUR CODE
        # 3. FC → ReLU → Dropout → FC
        logits = None     # YOUR CODE
        return logits

# Instantiate and sanity-check
model = TweetClassifier(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, n_classes=len(TOP_AUTHORS)).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

### Demo: class-weighted loss

Even after capping at 25K per class, the train split can still be slightly imbalanced. Mirror the `class_weight='balanced'` idea by passing inverse-frequency weights to `CrossEntropyLoss`.

In [ ]:
class_counts_train = np.bincount(y_train_ids, minlength=len(TOP_AUTHORS))
class_weights = len(y_train_ids) / (len(TOP_AUTHORS) * class_counts_train)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)
print("Class counts (train):", class_counts_train.tolist())
print("Class weights        :", np.round(class_weights, 3).tolist())

### Lab 6: Write the training loop

Write a PyTorch training loop that:

1. Creates `optimizer = torch.optim.Adam(model.parameters(), lr=LR)`.
2. Creates `loss_fn = nn.CrossEntropyLoss(weight=class_weights_t)`.
3. For each of `EPOCHS` epochs:
    - Train: for each batch `(ids, mask, y)`, do `zero_grad() → forward → loss → backward → step`. Accumulate train loss.
    - Validate: with `torch.no_grad()`, compute macro-F1 on val set.
    - Print the epoch summary.
4. Implement **manual early stopping**: track the best val F1 seen so far. If it hasn't improved for `PATIENCE` consecutive epochs, break. Save the best model's `state_dict()`.

You are given an `evaluate` helper that returns macro-F1 and the predictions.

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, targs = [], []
    with torch.no_grad():
        for ids, mask, y in loader:
            ids, mask = ids.to(device), mask.to(device)
            logits = model(ids, mask)
            preds.append(logits.argmax(dim=-1).cpu().numpy())
            targs.append(y.numpy())
    y_hat = np.concatenate(preds)
    y_true = np.concatenate(targs)
    return f1_score(y_true, y_hat, average='macro'), y_hat, y_true

In [ ]:
# 1. Optimizer and loss
optimizer = None  # YOUR CODE
loss_fn   = None  # YOUR CODE

# 2. Training loop with manual early stopping
best_val_f1 = 0.0
patience_counter = 0
best_state = None
history = {'train_loss': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    # ---- train phase ----
    model.train()
    total_loss, n_batches = 0.0, 0
    for ids, mask, y in train_loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        # YOUR CODE: zero_grad, forward, loss, backward, step
        pass
        n_batches += 1
    train_loss = total_loss / max(1, n_batches)

    # ---- validation phase ----
    val_f1, _, _ = evaluate(model, val_loader)

    history['train_loss'].append(train_loss)
    history['val_f1'].append(val_f1)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_macroF1={val_f1:.4f}")

    # YOUR CODE: early-stopping bookkeeping (track best_val_f1, best_state, patience_counter, break if >= PATIENCE)

### Lab 7: Evaluate on the test set

Load the best state dict, then run `evaluate` on `test_loader` and print a full classification report (with class names via `ID2LABEL`) plus a confusion matrix.

In [ ]:
# 1. Load best weights
# YOUR CODE: if best_state is not None, model.load_state_dict(best_state)

# 2. Evaluate on test set
test_f1, y_pred_nn, y_true_nn = None, None, None  # YOUR CODE

if y_pred_nn is not None:
    print(f"Neural model — test macro-F1: {test_f1:.4f}\n")
    target_names = [ID2LABEL[i] for i in range(len(TOP_AUTHORS))]
    print(classification_report(y_true_nn, y_pred_nn, target_names=target_names, digits=3))

## Section 6: Wrap-up — Which Model Would You Ship?

You've now built two classifiers on the same task:

- **Classical**: TF-IDF + LogReg (balanced). Usually 0.70-0.80 macro-F1.
- **Neural**: Embedding + masked mean-pool + MLP. Results will be similar here — the dataset is small and the tweets short.

Reflect:

1. Which model would you deploy to production, and why? Consider: accuracy, latency per prediction, memory, interpretability, retraining cadence.
2. What's the business cost of a misclassification? Routing an Amazon tweet to Apple means a customer waits 30 extra minutes. Routing a billing complaint to the general queue might lose a customer.
3. What would you try next to push past this baseline? (Hint: Module 2 — pretrained embeddings — is exactly about this.)

## Congratulations!

You have built your first end-to-end PyTorch text classifier on noisy real-world data, extracted labels from implicit metadata, and compared it head-to-head with a classical baseline. You are ready for Module 2: embeddings.